# Introduction

Do higher film budgets lead to more box office revenue? Let's find out if there's a relationship using the movie budgets and financial performance data that I've scraped from [the-numbers.com](https://www.the-numbers.com/movie/budgets) on **May 1st, 2018**. 

<img src=https://i.imgur.com/kq7hrEh.png>

# Import Statements

In [79]:
import pandas as pd
import matplotlib.pyplot as plt


# Notebook Presentation

In [80]:
pd.options.display.float_format = '{:,.2f}'.format

from pandas.plotting import register_matplotlib_converters
register_matplotlib_converters()

# Read the Data

In [81]:
data = pd.read_csv('cost_revenue_dirty.csv')


# Explore and Clean the Data

**Challenge**: Answer these questions about the dataset:
1. How many rows and columns does the dataset contain?
2. Are there any NaN values present?
3. Are there any duplicate rows?
4. What are the data types of the columns?

In [82]:
"""How many rows and cols?"""
print(f'row: {data.shape[0]}, cols: {data.shape[1]}')

"""Are there any NaN values present?"""
# print(data.isna())
print('Answer: There are no NaN values present in the data')

"""Are there any duplicate rows?"""
# print(data.duplicated())
print('There are no duplicate rows in the data')

"""What are the data types of the cols?"""
print(data.dtypes)

row: 5391, cols: 6
Answer: There are no NaN values present in the data
There are no duplicate rows in the data
Rank                      int64
Release_Date             object
Movie_Title              object
USD_Production_Budget    object
USD_Worldwide_Gross      object
USD_Domestic_Gross       object
dtype: object


### Data Type Conversions

**Challenge**: Convert the `USD_Production_Budget`, `USD_Worldwide_Gross`, and `USD_Domestic_Gross` columns to a numeric format by removing `$` signs and `,`. 
<br>
<br>
Note that *domestic* in this context refers to the United States.

In [83]:
data.USD_Production_Budget = data.USD_Production_Budget.str.replace('$', '')
data.USD_Production_Budget = data.USD_Production_Budget.str.replace(',', '')
data.USD_Production_Budget = pd.to_numeric(data.USD_Production_Budget)

data.USD_Worldwide_Gross = data.USD_Worldwide_Gross.str.replace('$', '')
data.USD_Worldwide_Gross = data.USD_Worldwide_Gross.str.replace(',', '')
data.USD_Worldwide_Gross = pd.to_numeric(data.USD_Worldwide_Gross)

data.USD_Domestic_Gross = data.USD_Domestic_Gross.str.replace('$', '')
data.USD_Domestic_Gross = data.USD_Domestic_Gross.str.replace(',', '')
data.USD_Domestic_Gross = pd.to_numeric(data.USD_Domestic_Gross)


**Challenge**: Convert the `Release_Date` column to a Pandas Datetime type. 

In [84]:
data.Release_Date = pd.to_datetime(data.Release_Date)
print(data.dtypes)

Rank                              int64
Release_Date             datetime64[ns]
Movie_Title                      object
USD_Production_Budget             int64
USD_Worldwide_Gross               int64
USD_Domestic_Gross                int64
dtype: object


### Descriptive Statistics

**Challenge**: 

1. What is the average production budget of the films in the data set?
2. What is the average worldwide gross revenue of films?
3. What were the minimums for worldwide and domestic revenue?
4. Are the bottom 25% of films actually profitable or do they lose money?
5. What are the highest production budget and highest worldwide gross revenue of any film?
6. How much revenue did the lowest and highest budget films make?

In [124]:
"""What is the average production budget of the films in the data set?"""
print(f'${round(data.USD_Production_Budget.mean())}')

"""What is the average worldwide gross revenue of films?"""
print(f'${round(data.USD_Worldwide_Gross.mean())}')

"""What are the minimums for worldwide and domestic revenue?"""
print(f'${data.USD_Worldwide_Gross.min()}')
print(f'${data.USD_Domestic_Gross.min()}')

"""Are the bottom 25% of films actually profitable or do they lose money?"""
bottom_25_num = round(data.shape[0]*.25)
data.sort_values(by='Rank')
bottom_movies = data[bottom_25_num:]


ww_gross = bottom_movies['USD_Worldwide_Gross'].mean()
dm_gross = bottom_movies['USD_Domestic_Gross'].mean()
pd_budget = bottom_movies['USD_Production_Budget'].mean()
result = pd_budget - (ww_gross+ dm_gross)

print(result)
print('The bottom 25% of films lose money')

"""What are the highest production budget and the highest worldwide gross revenue of any film?"""
print(f'Highest prod budget: {data['USD_Production_Budget'].max()}')
print(f'Highest gross revenue: {data['USD_Worldwide_Gross'].max()}')

"""How much revenue did the lowest and the highest budget films make?"""
data['Total Revenue'] = data['USD_Domestic_Gross'] + data['USD_Worldwide_Gross']
data_prod_sort = data.sort_values(by='USD_Production_Budget', ascending=False)

print(f'highest budget film:{data_prod_sort['Total Revenue'].iloc[0]}')

$31113738
$88855422
$0
$0
-100156994.64531288
The bottom 25% of films lose money
Highest prod budget: 425000000
Highest gross revenue: 2783918982
3544426607


# Investigating the Zero Revenue Films

**Challenge** How many films grossed $0 domestically (i.e., in the United States)? What were the highest budget films that grossed nothing?

**Challenge**: How many films grossed $0 worldwide? What are the highest budget films that had no revenue internationally?

### Filtering on Multiple Conditions

**Challenge**: Use the [`.query()` function](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.query.html) to accomplish the same thing. Create a subset for international releases that had some worldwide gross revenue, but made zero revenue in the United States. 

Hint: This time you'll have to use the `and` keyword.

### Unreleased Films

**Challenge**:
* Identify which films were not released yet as of the time of data collection (May 1st, 2018).
* How many films are included in the dataset that have not yet had a chance to be screened in the box office? 
* Create another DataFrame called data_clean that does not include these films. 

In [54]:
# Date of Data Collection
scrape_date = pd.Timestamp('2018-5-1')

### Films that Lost Money

**Challenge**: 
What is the percentage of films where the production costs exceeded the worldwide gross revenue? 

# Seaborn for Data Viz: Bubble Charts

### Plotting Movie Releases over Time

**Challenge**: Try to create the following Bubble Chart:

<img src=https://i.imgur.com/8fUn9T6.png>



# Converting Years to Decades Trick

**Challenge**: Create a column in `data_clean` that has the decade of the release. 

<img src=https://i.imgur.com/0VEfagw.png width=650> 

Here's how: 
1. Create a [`DatetimeIndex` object](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DatetimeIndex.html) from the Release_Date column. 
2. Grab all the years from the `DatetimeIndex` object using the `.year` property.
<img src=https://i.imgur.com/5m06Ach.png width=650>
3. Use floor division `//` to convert the year data to the decades of the films.
4. Add the decades as a `Decade` column to the `data_clean` DataFrame.

### Separate the "old" (before 1969) and "New" (1970s onwards) Films

**Challenge**: Create two new DataFrames: `old_films` and `new_films`
* `old_films` should include all the films before 1969 (up to and including 1969)
* `new_films` should include all the films from 1970 onwards
* How many films were released prior to 1970?
* What was the most expensive film made prior to 1970?

# Seaborn Regression Plots

**Challenge**: Use Seaborn's `.regplot()` to show the scatter plot and linear regression line against the `new_films`. 
<br>
<br>
Style the chart

* Put the chart on a `'darkgrid'`.
* Set limits on the axes so that they don't show negative values.
* Label the axes on the plot "Revenue in \$ billions" and "Budget in \$ millions".
* Provide HEX colour codes for the plot and the regression line. Make the dots dark blue (#2f4b7c) and the line orange (#ff7c43).

Interpret the chart

* Do our data points for the new films align better or worse with the linear regression than for our older films?
* Roughly how much would a film with a budget of $150 million make according to the regression line?

# Run Your Own Regression with scikit-learn

$$ REV \hat ENUE = \theta _0 + \theta _1 BUDGET$$

**Challenge**: Run a linear regression for the `old_films`. Calculate the intercept, slope and r-squared. How much of the variance in movie revenue does the linear model explain in this case?

# Use Your Model to Make a Prediction

We just estimated the slope and intercept! Remember that our Linear Model has the following form:

$$ REV \hat ENUE = \theta _0 + \theta _1 BUDGET$$

**Challenge**:  How much global revenue does our model estimate for a film with a budget of $350 million? 